# Finance RAG Assistant (Text Document Version)
This version avoids PDF parsing issues and loads readable text documents (.txt, .md, .csv, etc).

In [ ]:
!pip install openai chromadb gradio python-dotenv

In [ ]:
import os
import uuid
import chromadb
import gradio as gr
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    client = OpenAI(api_key=api_key)
else:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

In [ ]:
DOCUMENT_FOLDER = Path.home() / "Documents" / "finance_docs"
CHROMA_COLLECTION = "finance_knowledge_base" 

In [ ]:
def ensure_documents_folder():
    DOCUMENT_FOLDER.mkdir(parents=True, exist_ok=True)
    print("Place finance documents in:")
    print(DOCUMENT_FOLDER)

In [ ]:
def initialize_vector_db():
    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection(
        name=CHROMA_COLLECTION
    )
    return collection

In [ ]:
def load_documents():
    documents = []
    
    for file in DOCUMENT_FOLDER.iterdir():
        try:
            with open(file, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
                
            if text.strip():
                documents.append({
                    "text": text,
                    "source": file.name
                })
        except Exception:
            print("Skipping file:", file.name)
    
    print("Documents loaded:", len(documents))
    return documents

In [ ]:
def chunk_documents(documents, chunk_size=500):
    chunks = []
    
    for doc in documents:
        text = doc["text"]
        
        for i in range(0, len(text), chunk_size):
            chunk = text[i:i + chunk_size]
            
            chunks.append({
                "text": chunk,
                "source": doc["source"]
            })
    
    print("Chunks created:", len(chunks))
    return chunks

In [ ]:
def create_embeddings(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    
    embeddings = [item.embedding for item in response.data]
    return embeddings

In [ ]:
def store_vectors(chunks, embeddings, collection):
    ids = []
    documents = []
    
    for chunk in chunks:
        ids.append(str(uuid.uuid4()))
        documents.append(chunk["text"])
    
    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=ids
    )

In [ ]:
def build_vector_database():
    documents = load_documents()
    
    if not documents:
        print("No documents found")
        return
    
    chunks = chunk_documents(documents)
    texts = [chunk["text"] for chunk in chunks]
    
    embeddings = create_embeddings(texts)
    store_vectors(chunks, embeddings, collection)
    
    print("Vector database built successfully")

In [ ]:
def embed_query(query):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    
    return response.data[0].embedding

In [ ]:
def retrieve_context(query_vector, collection, k=3):
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k
    )
    
    context = "\n\n".join(results["documents"][0])
    return context

In [ ]:
SYSTEM_PROMPT = """
You are a financial analysis assistant.

Use the provided finance documents to answer the question.

If the answer is not in the documents,
say you do not know.
"""

In [ ]:
def generate_answer(question, context):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT + "\n\nContext:\n" + context
        },
        {
            "role": "user",
            "content": question
        }
    ]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    return response.choices[0].message.content

In [ ]:
def rag_pipeline(question):
    query_vector = embed_query(question)
    context = retrieve_context(query_vector, collection)
    answer = generate_answer(question, context)
    return answer

In [ ]:
def chat(message, history):
    answer = rag_pipeline(message)
    return answer

In [ ]:
ensure_documents_folder()
collection = initialize_vector_db()
build_vector_database()

In [ ]:
view = gr.ChatInterface(
    chat,
    type="messages"
).launch(inbrowser=True)